# StackSage — Retrieval Experiments
Compare BM25, Dense, and Hybrid+Reranking on a sample of ground-truth questions.

In [ ]:
import sys, json
sys.path.insert(0,'..') 
from dotenv import load_dotenv; load_dotenv('../.env')
from ingestion.embed_and_index import get_qdrant, get_es, get_embedder
from rag.retriever import HybridRetriever
from rag.reranker import CrossEncoderReranker

retriever = HybridRetriever(get_qdrant(), get_es(), get_embedder())
reranker  = CrossEncoderReranker()
print('Clients ready')

## Load Ground Truth

In [ ]:
GT_PATH = '../data/eval/ground_truth.jsonl'
gt = [json.loads(l) for l in open(GT_PATH) if l.strip()]
print(f'Ground truth questions: {len(gt)}')
print(gt[0]['question_title'])

## Single-Query Comparison

In [ ]:
QUERY = gt[0]['question_title']
GOLD  = gt[0]['question_id']
print('Query:', QUERY)
print('Gold ID:', GOLD)

bm25   = retriever.search_bm25(QUERY,[],5,None,10)
dense  = retriever.search_dense(QUERY,[],5,None,10)
fused  = retriever.reciprocal_rank_fusion(bm25,dense)
ranked = reranker.rerank(QUERY, fused[:20], top_k=5)

for method,docs in [('BM25',bm25),('Dense',dense),('Hybrid',fused[:5]),('Reranked',ranked)]:
    ids   = [d.get('question_id') for d in docs[:5]]
    found = GOLD in ids
    rank  = ids.index(GOLD)+1 if found else None
    print(f'{method:<10} found={found}  rank={rank}')

## Batch Evaluation (n=50)

In [ ]:
from evaluation.evaluate import evaluate_retrieval
results = evaluate_retrieval('../data/eval/ground_truth.jsonl', retriever, n=50)
import pandas as pd
df = pd.DataFrame(results).T
print(df.to_string())

## Bar Chart

In [ ]:
import matplotlib.pyplot as plt
df[['hit_rate_at_5','hit_rate_at_10','mrr']].plot(
    kind='bar', figsize=(10,5), title='Retrieval Method Comparison')
plt.ylabel('Score'); plt.xticks(rotation=0); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## RRF Deep-Dive
Inspect score contributions from each list.

In [ ]:
fused_top10 = retriever.reciprocal_rank_fusion(bm25,dense)[:10]
for i,d in enumerate(fused_top10,1):
    print(f"{i:2}. rrf={d.get('retrieval_score',0):.4f}  qid={d.get('question_id')}  {d.get('question_title','')[:60]}")

## Embedding Similarity Sanity Check

In [ ]:
from sentence_transformers import SentenceTransformer, util
emb   = get_embedder()
vec_q = emb.encode(QUERY, normalize_embeddings=True)
for d in dense[:3]:
    vec_d = emb.encode(d.get('question_title',''), normalize_embeddings=True)
    sim   = float(util.cos_sim(vec_q, vec_d))
    print(f'sim={sim:.4f}  {d.get("question_title","")[:70]}')